# Lab 06 · MPI Heat 2D · domain decomposition and halo exchange

You have the serial stencil (lab 01), a roofline (lab 02), OpenMP threading (lab 03-04), and the MPI primitives (lab 05). Now build the real thing: a **2D domain decomposition** of the heat stencil with a **halo exchange** each step, running across multiple Crux nodes.

**Prerequisites.** Lab 05 (MPI basics). The math is unchanged from lab 01.

**Builds toward.** Lab 07 (hybrid MPI+OpenMP: MPI across nodes, OpenMP within each rank).

> **📚 Where to look when you're stuck**
>
> - [**MPI Cartesian topology**](https://www.mpich.org/static/docs/latest/www3/MPI_Cart_create.html)
> - [**MPI 4.1 standard**](https://www.mpi-forum.org/docs/)
> - [**Halo exchange patterns**](https://enccs.github.io/intermediate-mpi/) (ENCCS tutorial)



## How this notebook works

Same three surfaces as lab 01 and 02: **[Hub]**, **[Hub -> Crux]**, **[Crux compute]**.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER; re-run.
env = setupLab(labName="lab06", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + prerequisite artifact.
preflight([
    check("passwordless ssh", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab06 dir on cluster", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing"),
    check("lab05 mpiHello binary", remoteFileExists(env['HPC_LAB_DIR'].replace('lab06','lab05') + '/mpiHello')),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')),
             ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> Crux] Make the lab dir if missing.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab06 dir ready')


## Part 1 · The decomposition

Split the NxN grid across a PxQ grid of MPI ranks. Each rank owns a `(N/P) x (N/Q)` local tile, plus a one-cell **halo** on each side holding a copy of the neighbor's edge. Each timestep:

1. Exchange halos with the 4 neighbors (send my edges, receive theirs)
2. Compute the stencil on the tile
3. Repeat

The **surface-to-volume** ratio matters: for large tiles, comm/compute is small; for small tiles, comm dominates. That's why weak scaling (fixed tile size, N grows with #ranks) is often preferred over strong scaling for this kind of code.


In [ ]:
# [Hub] The full MPI+halo source is big; ship it as a whole file.
src = '''
/* heat2Dmpi.c - 2D domain decomposition of the heat stencil.
 * Uses MPI Cartesian communicator + non-blocking halo exchange.
 * Same physics/timestep as lab01 heat2D.c. */
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
static double wall(void){struct timespec t;clock_gettime(CLOCK_MONOTONIC,&t);return t.tv_sec+t.tv_nsec*1e-9;}
static double ic(int gi, int gj, int N){double x=(double)gi/N,y=(double)gj/N;
    return (x>0.4&&x<0.6&&y>0.4&&y<0.6)?1.0:0.0;}
int main(int argc, char **argv){
    MPI_Init(&argc,&argv);
    int rank,size; MPI_Comm_rank(MPI_COMM_WORLD,&rank);
    MPI_Comm_size(MPI_COMM_WORLD,&size);
    int N=1024, steps=500;
    for(int i=1;i<argc;i++){
        if(!strcmp(argv[i],"--N")&&i+1<argc) N=atoi(argv[++i]);
        if(!strcmp(argv[i],"--steps")&&i+1<argc) steps=atoi(argv[++i]);
    }
    int dims[2]={0,0}; MPI_Dims_create(size,2,dims);
    int periods[2]={0,0}; MPI_Comm cart;
    MPI_Cart_create(MPI_COMM_WORLD,2,dims,periods,1,&cart);
    int coord[2]; MPI_Cart_coords(cart,rank,2,coord);
    int nUp,nDown,nLeft,nRight;
    MPI_Cart_shift(cart,0,1,&nUp,&nDown);
    MPI_Cart_shift(cart,1,1,&nLeft,&nRight);
    int lx=N/dims[0], ly=N/dims[1];
    /* +2 in each dim for halos */
    double *u    = calloc((lx+2)*(ly+2), sizeof(double));
    double *unew = calloc((lx+2)*(ly+2), sizeof(double));
    /* first-touch init */
    for(int i=1;i<=lx;i++) for(int j=1;j<=ly;j++){
        int gi=coord[0]*lx+(i-1), gj=coord[1]*ly+(j-1);
        u[i*(ly+2)+j] = ic(gi,gj,N);
    }
    double alpha=0.1, dt=0.24, h=1.0;
    /* strided column types for left/right sends */
    MPI_Datatype colType;
    MPI_Type_vector(lx,1,ly+2,MPI_DOUBLE,&colType); MPI_Type_commit(&colType);
    double t0 = wall();
    for(int s=0;s<steps;s++){
        MPI_Request req[8]; int nr=0;
        /* rows: send row 1 up, receive top halo; send row lx down, receive bottom halo */
        MPI_Isend(&u[1*(ly+2)+1],       ly, MPI_DOUBLE, nUp,   0, cart, &req[nr++]);
        MPI_Irecv(&u[0*(ly+2)+1],       ly, MPI_DOUBLE, nUp,   0, cart, &req[nr++]);
        MPI_Isend(&u[lx*(ly+2)+1],      ly, MPI_DOUBLE, nDown, 0, cart, &req[nr++]);
        MPI_Irecv(&u[(lx+1)*(ly+2)+1], ly, MPI_DOUBLE, nDown, 0, cart, &req[nr++]);
        /* cols: strided */
        MPI_Isend(&u[1*(ly+2)+1],       1, colType,     nLeft, 0, cart, &req[nr++]);
        MPI_Irecv(&u[1*(ly+2)+0],       1, colType,     nLeft, 0, cart, &req[nr++]);
        MPI_Isend(&u[1*(ly+2)+ly],      1, colType,     nRight,0, cart, &req[nr++]);
        MPI_Irecv(&u[1*(ly+2)+ly+1],    1, colType,     nRight,0, cart, &req[nr++]);
        MPI_Waitall(nr, req, MPI_STATUSES_IGNORE);
        for(int i=1;i<=lx;i++) for(int j=1;j<=ly;j++){
            int k=i*(ly+2)+j;
            unew[k] = u[k] + alpha*dt/(h*h) *
                     (u[k-1]+u[k+1]+u[k-(ly+2)]+u[k+(ly+2)] - 4.0*u[k]);
        }
        double *tmp=u; u=unew; unew=tmp;
    }
    double dt2 = wall() - t0;
    double totalLupd = (double)N*N*steps;
    double localSum=0.0, globalSum=0.0;
    for(int i=1;i<=lx;i++) for(int j=1;j<=ly;j++) localSum += u[i*(ly+2)+j];
    MPI_Reduce(&localSum,&globalSum,1,MPI_DOUBLE,MPI_SUM,0,cart);
    if(rank==0){
        double mlups = totalLupd/dt2/1e6;
        printf("MPI heat2D: ranks=%d dims=%dx%d N=%d steps=%d wall=%.3fs mlups=%.2f sumU=%.6e\\n",
               size, dims[0], dims[1], N, steps, dt2, mlups, globalSum);
    }
    MPI_Type_free(&colType);
    free(u); free(unew);
    MPI_Finalize(); return 0;
}
'''
(labDir/'heat2Dmpi.c').write_text(src)
showFile(labDir/'heat2Dmpi.c', language='c', maxLines=25, title='heat2Dmpi.c (top)')


In [ ]:
checkpoint("Part 1 - MPI source", [
    check("heat2Dmpi.c present", fileExists(str(labDir/'heat2Dmpi.c'))),
    check("uses Cart_create",
          fileContains(str(labDir/'heat2Dmpi.c'), 'MPI_Cart_create')),
])


## Part 2 · Build and run on 4 nodes

Build once, launch on 4 nodes with 8 ranks per node (32 ranks total). Watch the output for the aggregate MLUP/s.


In [ ]:
# [Hub -> Crux] Build + run on 4 nodes x 8 ranks.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -O3 -o heat2Dmpi heat2Dmpi.c
NRANKS_PER_NODE=8
NNODES=$(wc -l < $PBS_NODEFILE); NRANKS=$(( NNODES * NRANKS_PER_NODE ))
mpiexec -n $NRANKS --ppn $NRANKS_PER_NODE ./heat2Dmpi --N 1024 --steps 500
'''
pbsPath = labDir/'mpiJob.pbs'
pbsPath.write_text(pbsHeader(name='lab06MPI', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             select='4:system=crux', walltime='00:15:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/mpi.out') + jobBody)
sshPut(str(labDir/'heat2Dmpi.c'), env['HPC_LAB_DIR']+'/heat2Dmpi.c')
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/mpiJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/mpiJob.pbs'); waitJob(jobID, 30, 1500)
sshGet(env['HPC_LAB_DIR']+'/mpi.out', str(labDir/'mpi.out'))
print((labDir/'mpi.out').read_text())


In [ ]:
checkpoint("Part 2 - MPI heat2D on 4 nodes", [
    check("MPI output present", fileExists(str(labDir/'mpi.out'))),
    check("mlups printed", fileContains(str(labDir/'mpi.out'), 'mlups')),
])


## Part 3 · Strong scaling · 1, 2, 4, 8 nodes

Fix the problem size, vary the number of ranks. Expect near-ideal speedup until communication overhead eats the gain.


In [ ]:
# [Hub -> Crux] Strong scale from 1 to 8 nodes.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
for nn in 1 2 4 8; do
  NRANKS=$(( nn * 8 ))
  mpiexec -n $NRANKS --ppn 8 ./heat2Dmpi --N 2048 --steps 200 | tee -a strong.log
done
cat strong.log
'''
pbsPath = labDir/'strongJob.pbs'
pbsPath.write_text(pbsHeader(name='lab06Strong', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             select='8:system=crux', walltime='00:30:00',
                             filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/strong.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/strongJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/strongJob.pbs'); waitJob(jobID, 30, 2400)
sshGet(env['HPC_LAB_DIR']+'/strong.out', str(labDir/'strong.out'))
print((labDir/'strong.out').read_text())


In [ ]:
# [Hub] Parse and plot.
import re, pandas as pd
rows = []
for line in open(labDir/'strong.out'):
    m = re.search(r'ranks=(\d+).*wall=([\d.]+)s mlups=([\d.]+)', line)
    if m: rows.append({'variant':'mpi','ranks':int(m.group(1)),
                       'threads':1,'N':2048,'steps':200,
                       'wall_s':float(m.group(2)),'mlups':float(m.group(3)),
                       'lab':6,'io_s':0})
df = pd.DataFrame(rows)
df.to_csv(labDir/'timings.csv', index=False)
print(df.to_string(index=False))
r = plotScaling(str(labDir/'timings.csv'), kind='strong', variantFilter='mpi',
                baselineCol='ranks', timeCol='wall_s',
                outPath=str(labDir/'figures'/'mpiStrong'))
print('wrote:', [str(p) for p in r])


In [ ]:
checkpoint("Part 3 - strong scaling", [
    check("strong scaling plot", fileExists(str(labDir/'figures'/'mpiStrong.pdf'))),
])


## Part 4 · Weak scaling · tile size fixed

Fix the tile per rank (say 512x512); grow N so bigger jobs get bigger problems. **Ideal weak scaling is flat efficiency at 1.0.** Deviations tell you where communication starts to hurt.


In [ ]:
# [Hub] Placeholder cell - lab 11 does the full weak-scaling sweep; here we
# note the pattern so students can see it.
showNote('Weak scaling recipe (run in lab 11 as the semester project):\n'
        '  for nn in 1 2 4 8 16 32:\n'
        '      N = 512 * int(sqrt(nn * 8))     # scale N with sqrt(ranks)\n'
        '      run ./heat2Dmpi --N $N --steps 200 on nn nodes', kind='info')


In [ ]:
checkpoint("Part 4 - weak scaling recipe", [
    check("strong scaling exists",
          fileExists(str(labDir/'figures'/'mpiStrong.pdf'))),
])


## Part 5 · Correctness · sum of u

The final `sumU` printed by the MPI run should match the serial reference (from lab 01) to floating-point noise. If it doesn't, either your halo exchange is broken (missing an edge) or the initial condition is off (wrong global indexing).


In [ ]:
# [Hub] Compare sumU from MPI run to a serial reference.
import re
txt = (labDir/'mpi.out').read_text()
m = re.search(r'sumU=([\d.eE+-]+)', txt)
sumMPI = float(m.group(1)) if m else None
print(f'MPI sumU: {sumMPI}')
print('Compare against lab 01 science.csv last row - should agree to ~1e-10 relative.')


In [ ]:
checkpoint("Part 5 - correctness", [
    check("MPI sumU printed", lambda: (sumMPI is not None, str(sumMPI))),
])


## Part 6 · Bridge to lab 07

Now you have MPI across nodes. The next step is **hybrid**: MPI between nodes, OpenMP within each rank. That combination is how virtually every modern HPC application scales — MPI hides the network, OpenMP exploits the shared memory of each node.


## Wrap up

Moved the spine forward one lab. Ready for the next.


### Lab scorecard


In [ ]:
labSummary("MPI Heat 2D")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("MPI Heat 2D")
